In [48]:
import stim
import stim
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from wrappers.stim_wrapper import (create_circuit_polar_stim_normal, convert_i_to_meas_type, 
simulate_batch_and_save_result_polar_normal)

from wrappers.polar_wrapper import is_q1prep_accepted, get_logical_error_on_accepted_states, get_q1prep_accepted_states

import torch
import torch.nn.functional as F

import collections
from wrappers.stim_wrapper import simulate_stim_polar_code_normal
from wrappers.polar_wrapper import get_logical_error_on_accepted_states_SCL

from Decoders.SCL import PyDecoderPolarSCL 


In [ ]:
n = 4

shots = 10000000
seed = 123456
decoder_type = "anqi"
accept_all = True

i_list = [6, 7]
lstate_list = ["x", "z"]
sim_type_list = ["normal", "m1"]
p_error_list = [0.01, 0.0075, 0.005, 0.0025, 0.001]

for i in i_list:
    zpos_list = [-1, -1, 1, 3, 6, 7, 22, 15, 90, 31, 362]
    zpos_list[n] = i-1
    for lstate in lstate_list:
        for sim_type in sim_type_list:
            for p_error in p_error_list:
                # i = 6
                # lstate = "x"
                # sim_type = "m1"
                # p_error = 0.005


                results = simulate_stim_polar_code_normal(n, lstate, sim_type, i, p_error, shots, seed)

                # updating the counter with the new results
                current_counter = collections.Counter()
                current_counter.update(results)

                # this is for decoding the result
                meas_type = convert_i_to_meas_type(i, n, lstate)

                counts = dict(current_counter)
                total_shots = sum(counts.values())

                zpos_list = [-1, -1, 1, 3, 6, 7, 22, 15, 90, 31, 362]
                zpos_list[n] = i - 1

                frozen_bits_mask = np.ones(2**n, dtype=np.int32)
                frozen_bits_mask[i-1] = 0
                frozen_bits_mask = list(frozen_bits_mask)

                if sim_type == "m1":
                    circuit_type = "proposal"
                else:
                    circuit_type = "baseline"

                save_dataset_path = f"dataset_GNN/polar_{circuit_type}_n{n}_i{i}_{lstate}_p{p_error}.npz"

                decoder = PyDecoderPolarSCL(1, 2**n, 1, frozen_bits_mask)
                count_accept, count_logerror, count_undecided, ler, detect_normal, decoding_normal, count_tp, count_fp, count_fn, count_tn = \
                        get_logical_error_on_accepted_states_SCL(
                            n, lstate.upper(), counts, decoder, p_error,zpos_list, True, save_dataset_path, sim_type
                        )
                print(count_accept, count_logerror, count_undecided, 1-ler, detect_normal, decoding_normal) 
                print(count_tp, count_fp, count_fn, count_tn, (count_tp + count_fn) / shots, (count_fp + count_tn) / (count_tp + count_fn), (count_fp + count_tn) / count_tp)


frozen_bits: 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 


In [59]:
count_accept / shots

0.677772

In [47]:
load_dataset_path = f"dataset_GNN/polar_{circuit_type}_n{n}_i{i}_{lstate}_p{p_error}.npz"
data = np.load(load_dataset_path)
data

NpzFile 'dataset_GNN/polar_proposal_n3_i4_x_p0.01.npz' with keys: X, data_bits, frame, tp, fp...

In [42]:
data["X"][0], data["data_bits"][0]

(array([0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1], dtype=uint8),
 array([1, 1, 0, 0, 0, 1, 0, 1], dtype=uint8))

In [45]:
data["sim_type"]

array('m1', dtype='<U2')